In [45]:
!pip install geopandas shapely


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [47]:
# Chemins vers les dossiers contenant les fichiers CSV
base_path = r'data\open_data_millesime_2025-07-a_dep31_csv\csv\\'

In [48]:
try:
    # 1. Chargement avec séparateur ';'
    print("Chargement de batiment_groupe...")
    df_geo = pd.read_csv(base_path + 'batiment_groupe.csv', 
                         sep=';', 
                         usecols=['geom_groupe','batiment_groupe_id', 'code_epci_insee', 'libelle_commune_insee'])

    print("Chargement de dpe_representatif...")
    df_dpe = pd.read_csv(base_path + 'batiment_groupe_dpe_representatif_logement.csv', 
                         sep=';', 
                         usecols=['batiment_groupe_id', 'periode_construction_dpe', 'annee_construction_dpe', 'type_batiment_dpe'])

    print("Chargement de rpls (parc social)...")
    df_rpls = pd.read_csv(base_path + 'batiment_groupe_rpls.csv', 
                          sep=';', 
                          usecols=['batiment_groupe_id', 'nb_log', 's_log_hab', 'type_construction', 'nb_classe_ener_a',
                                   'nb_classe_ener_b','nb_classe_ener_c','nb_classe_ener_d','nb_classe_ener_e',
                                   'nb_classe_ener_f','nb_classe_ener_g','nb_classe_ener_nc','classe_ener_principale', 'l_annee_construction'])

    # 2. Fusion (Jointures)
    # On garde tous les bâtiments qui sont dans le fichier social (RPLS)
    df_merged = pd.merge(df_rpls, df_geo, on='batiment_groupe_id', how='left')
    
    # On ajoute les infos DPE
    df_final = pd.merge(df_merged, df_dpe, on='batiment_groupe_id', how='left')

    # 3. Aperçu du résultat
    print(f"\nSuccès ! Le fichier final contient {len(df_final)} lignes.")
    display(df_final.head())

    # 4. Sauvegarde pour ton Streamlit
    df_final.to_csv(r'data\data_social_31.csv', index=False, sep=';')
    print("\nFichier 'data_social_31.csv' créé avec succès.")

except Exception as e:
    print(f"Erreur lors du traitement : {e}")

Chargement de batiment_groupe...
Chargement de dpe_representatif...
Chargement de rpls (parc social)...

Succès ! Le fichier final contient 7595 lignes.


,batiment_groupe_id,nb_classe_ener_a,nb_classe_ener_b,nb_classe_ener_c,nb_classe_ener_d,nb_classe_ener_e,nb_classe_ener_f,nb_classe_ener_g,nb_classe_ener_nc,classe_ener_principale,l_annee_construction,nb_log,s_log_hab,type_construction,geom_groupe,libelle_commune_insee,code_epci_insee,type_batiment_dpe,periode_construction_dpe,annee_construction_dpe
0,bdnb-bg-1115-G1VZ-Q3TM,3.0,1.0,0.0,0.0,0.0,0,0,0,A,"[ ""2015"" ]",4.0,272.0,"[ ""Collectif"" ]","MULTIPOLYGON (((576360.7 6280468.2,576354.5 62...",Toulouse,243100518,appartement,2013-2021,NaN
1,bdnb-bg-1133-KXCL-DZT5,0.0,0.0,0.0,3.0,0.0,0,0,0,D,"[ ""2008"" ]",3.0,131.0,"[ ""Collectif"" ]","MULTIPOLYGON (((568235.5 6277954.7,568228.3 62...",Tournefeuille,243100518,appartement,2006-2012,2006.0
2,bdnb-bg-115H-SYCX-D7UB,0.0,0.0,0.0,1.0,0.0,0,0,0,D,"[ ""1996"" ]",1.0,84.0,"[ ""Individuel"" ]","MULTIPOLYGON (((568159.4 6299251.6,568158.1 62...",Castelnau-d'Estrétefonds,200034957,maison,1989-2000,1996.0
3,bdnb-bg-118C-PUXU-X8T8,0.0,16.0,0.0,0.0,0.0,0,0,0,B,"[ ""1949"" ]",16.0,1100.0,"[ ""Collectif"" ]","MULTIPOLYGON (((571673.3 6280702.8,571677.7 62...",Toulouse,243100518,immeuble,1948-1974,1949.0
4,bdnb-bg-118N-VC9X-DWQX,0.0,0.0,1.0,0.0,0.0,0,0,0,C,"[ ""1994"" ]",1.0,85.0,"[ ""Individuel"" ]","MULTIPOLYGON (((563567.3 6281250.2,563568.5 62...",Colomiers,243100518,maison,1989-2000,1994.0


Erreur lors du traitement : [Errno 13] Permission denied: 'data\\data_social_31.csv'


In [49]:
# On recharge le fichier qu'on vient de créer pour vérifier
test_df = pd.read_csv(r'data\data_social_31.csv', sep=';')
print(f"Colonnes après rechargement : {test_df.columns.tolist()}")

Colonnes après rechargement : ['batiment_groupe_id', 'nb_classe_ener_a', 'nb_classe_ener_b', 'nb_classe_ener_c', 'nb_classe_ener_d', 'nb_classe_ener_e', 'nb_classe_ener_f', 'nb_classe_ener_g', 'nb_classe_ener_nc', 'classe_ener_principale', 'l_annee_construction', 'nb_log', 's_log_hab', 'type_construction', 'geom_groupe', 'libelle_commune_insee', 'code_epci_insee', 'periode_construction_dpe', 'annee_construction_dpe']


In [50]:
# 1. Chargement des données en Lambert-93 (EPSG:2154)
df_final['geometry'] = df_final['geom_groupe'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df_final, geometry='geometry', crs="EPSG:2154")

# 2. Calcul du centroïde pour chaque bâtiment (en Lambert-93)
# Crée deux colonnes temporaires pour les coordonnées projetées
gdf['centroid_tmp'] = gdf.geometry.centroid

# 3. Conversion vers WGS84 (EPSG:4326) pour Folium
gdf = gdf.to_crs(epsg=4326)
gdf['centroid_tmp'] = gdf['centroid_tmp'].to_crs(epsg=4326)

# 4. Extraction LAT/LON (Maintenant que c'est en degrés)
gdf['lat'] = gdf['centroid_tmp'].y
gdf['lon'] = gdf['centroid_tmp'].x
# Supprime la colonne temporaire pour ne pas alourdir le fichier
gdf = gdf.drop(columns=['centroid_tmp'])

# 5. Sauvegarde
gdf.to_file(r"data\data_social_31_geo.geojson", driver='GeoJSON')
print("Fichier sauvegardé avec Lat/Lon précises !")

Fichier sauvegardé avec Lat/Lon précises !
